In [1]:
#April 22, 2026
#Charlotte Meyer
#Data Seminar
#This code is meant to use statsmodels to create a logistic regression

In [2]:
#importing packages
import statsmodels.api
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
import numpy as np
#import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split

In [3]:
#reading in the data 
cejst = pd.read_csv("cejst_logistic_regression.csv", index_col=0, low_memory=False)

#setting pandas columns so I can see all of them
pd.set_option('display.max_columns', None)

In [4]:
#making a function that will split columns into bins for fixed effects

def split_into_bins(fixed_effect_column):

    #cut into 11 bins (arbitrary)
    #using cut, NOT qcut
    cejst[fixed_effect_column + "_bin"] = pd.cut(cejst[fixed_effect_column], 11, labels=range(1,12))
    
    #creating dummies, prefix adds column in front of bin name
    dummies = pd.get_dummies(cejst[fixed_effect_column + "_bin"])#, prefix=fixed_column_effect)
    
    #removing middle dummy
    dummies = dummies[[i for i in dummies.columns if i != 6]]
    
    #renaming dummies
    dummies.columns = [fixed_effect_column + "_" + str(i) for i in dummies.columns ]
    
    #combining existing columns with cejst
#    dummies[fixed_effect_column] = cejst[fixed_effect_column]
#    dummies['has_datacenter'] = cejst['has_datacenter']

    #drop na
    dummies = dummies.dropna()
    
    #viewing bins
    return dummies
#    return cejst.groupby('bin')[[fixed_effect_column, 'has_datacenter']].mean() #this shows the bins; will show bin 6

In [5]:
linguistic_isolation_dummies = split_into_bins("linguistic_isolation")
linguistic_isolation_dummies

#cejst = cejst.copy()
cejst = pd.concat([cejst, linguistic_isolation_dummies], axis=1)

/var/folders/sl/dyq7sj0x2_z0dr2x82v1ln_80000gn/T/ipykernel_34180/23027534.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cejst[fixed_effect_column + "_bin"] = pd.cut(cejst[fixed_effect_column], 11, labels=range(1,12))


In [6]:
perc_poverty_dummies = split_into_bins("perc_poverty")
perc_poverty_dummies

cejst = pd.concat([cejst, perc_poverty_dummies], axis=1)

/var/folders/sl/dyq7sj0x2_z0dr2x82v1ln_80000gn/T/ipykernel_34180/23027534.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cejst[fixed_effect_column + "_bin"] = pd.cut(cejst[fixed_effect_column], 11, labels=range(1,12))


In [7]:
cejst

,GEOID,County Name,State,perc_black,perc_native_american,perc_asian,perc_hawaiin_pacific,perc_mixed_race,perc_white,perc_hispanic,Percent other races,perc_children,Percent age 10 to 64,perc_elderly,Total threshold criteria exceeded,Total categories exceeded,Identified as disadvantaged without considering neighbors,Identified as disadvantaged based on neighbors and relaxed low income threshold only,Identified as disadvantaged due to tribal overlap,Identified as disadvantaged,perc_disadvantaged,Share of neighbors that are identified as disadvantaged,Identified as disadvantaged in v1.0,Identified as disadvantaged solely due to status in v1.0 (grandfathered),population,Interpolated number of off-campus students in poverty,Adjusted percent of individuals below 200% Federal Poverty Line (percentile),perc_poverty,Is low income?,Income data has been estimated based on geographic neighbor income,Greater than or equal to the 90th percentile for expected agriculture loss rate and is low income?,Expected agricultural loss rate (Natural Hazards Risk Index) (percentile),Expected agricultural loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected building loss rate and is low income?,Expected building loss rate (Natural Hazards Risk Index) (percentile),Expected building loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected population loss rate and is low income?,Expected population loss rate (Natural Hazards Risk Index) (percentile),loss_rate,Share of properties at risk of flood in 30 years (percentile),flood_risk,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years and is low income?,Share of properties at risk of fire in 30 years (percentile),Share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years and is low income?,Greater than or equal to the 90th percentile for energy burden and is low income?,Energy burden (percentile),energy_burden,Greater than or equal to the 90th percentile for PM2.5 exposure and is low income?,PM2.5 in the air (percentile),pm_air,Greater than or equal to the 90th percentile for diesel particulate matter and is low income?,Diesel particulate matter exposure (percentile),diesel_exposure,Greater than or equal to the 90th percentile for traffic proximity and is low income?,Traffic proximity and volume (percentile),traffic_prox,Greater than or equal to the 90th percentile for DOT transit barriers and is low income?,travel_barriers,Greater than or equal to the 90th percentile for housing burden and is low income?,Housing burden (percent) (percentile),housing_burden,"Greater than or equal to the 90th percentile for lead paint, the median house value is less than 90th percentile and is low income?",Percent pre-1960s housing (lead paint indicator) (percentile),Percent pre-1960s housing (lead paint indicator),Median value ($) of owner-occupied housing units (percentile),house_value,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent and is low income?,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent,Share of the tract's land area that is covered by impervious surface or cropland as a percent,pavement_cropland,Does the tract have at least 35 acres in it?,Tract experienced historic underinvestment and remains low income,Tract experienced historic underinvestment,no_plumbing,Share of homes with no kitchen or indoor plumbing (percent),Greater than or equal to the 90th percentile for proximity to hazardous waste facilities and is low income?,

In [8]:
# Define predictor variables
predictors = ['perc_white', 'perc_poverty', 'pm_air', 'perc_unemployment', 
              'hazard_waste_prox', 'life_expectancy', 'perc_hispanic', 
              'perc_elderly', 'perc_children', 'perc_disadvantaged', 
              'population', 'loss_rate', 'flood_risk', 'energy_burden', 
              'diesel_exposure', 'traffic_prox', 'travel_barriers', 
              'housing_burden', 'house_value', 'pavement_cropland', 
              'no_plumbing', 'superfund_prox', 
              'asthma', 'diabetes', 'heart_disease', 'household_income_compared', 
              'linguistic_isolation', 'no_hs_degree']

print(cejst[predictors].isnull().sum())
#removing wastewater_discharge from predictors because it has 20,000 NA's

perc_white                    744
perc_poverty                  158
pm_air                       1875
perc_unemployment             973
hazard_waste_prox             158
life_expectancy              6986
perc_hispanic                 744
perc_elderly                  861
perc_children                 861
perc_disadvantaged              0
population                     28
loss_rate                    1604
flood_risk                    796
energy_burden                1054
diesel_exposure               739
traffic_prox                 2834
travel_barriers              1801
housing_burden               1158
house_value                  2201
pavement_cropland            1595
no_plumbing                  1158
superfund_prox                158
asthma                       3821
diabetes                     3821
heart_disease                3821
household_income_compared    5021
linguistic_isolation         1932
no_hs_degree                  879
dtype: int64


In [9]:
#dropping NA's; code is from Claude


#dropping all rows with NaN in predictor columns
cejst = cejst.dropna(subset=predictors)

#verify they match
print(f"X shape: {cejst[predictors].shape}, y shape: {cejst["has_datacenter"].shape}")
#verify no na's
print(cejst[predictors].isnull().sum())

X shape: (59078, 28), y shape: (59078,)
perc_white                   0
perc_poverty                 0
pm_air                       0
perc_unemployment            0
hazard_waste_prox            0
life_expectancy              0
perc_hispanic                0
perc_elderly                 0
perc_children                0
perc_disadvantaged           0
population                   0
loss_rate                    0
flood_risk                   0
energy_burden                0
diesel_exposure              0
traffic_prox                 0
travel_barriers              0
housing_burden               0
house_value                  0
pavement_cropland            0
no_plumbing                  0
superfund_prox               0
asthma                       0
diabetes                     0
heart_disease                0
household_income_compared    0
linguistic_isolation         0
no_hs_degree                 0
dtype: int64


In [10]:
cejst

,GEOID,County Name,State,perc_black,perc_native_american,perc_asian,perc_hawaiin_pacific,perc_mixed_race,perc_white,perc_hispanic,Percent other races,perc_children,Percent age 10 to 64,perc_elderly,Total threshold criteria exceeded,Total categories exceeded,Identified as disadvantaged without considering neighbors,Identified as disadvantaged based on neighbors and relaxed low income threshold only,Identified as disadvantaged due to tribal overlap,Identified as disadvantaged,perc_disadvantaged,Share of neighbors that are identified as disadvantaged,Identified as disadvantaged in v1.0,Identified as disadvantaged solely due to status in v1.0 (grandfathered),population,Interpolated number of off-campus students in poverty,Adjusted percent of individuals below 200% Federal Poverty Line (percentile),perc_poverty,Is low income?,Income data has been estimated based on geographic neighbor income,Greater than or equal to the 90th percentile for expected agriculture loss rate and is low income?,Expected agricultural loss rate (Natural Hazards Risk Index) (percentile),Expected agricultural loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected building loss rate and is low income?,Expected building loss rate (Natural Hazards Risk Index) (percentile),Expected building loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected population loss rate and is low income?,Expected population loss rate (Natural Hazards Risk Index) (percentile),loss_rate,Share of properties at risk of flood in 30 years (percentile),flood_risk,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years and is low income?,Share of properties at risk of fire in 30 years (percentile),Share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years and is low income?,Greater than or equal to the 90th percentile for energy burden and is low income?,Energy burden (percentile),energy_burden,Greater than or equal to the 90th percentile for PM2.5 exposure and is low income?,PM2.5 in the air (percentile),pm_air,Greater than or equal to the 90th percentile for diesel particulate matter and is low income?,Diesel particulate matter exposure (percentile),diesel_exposure,Greater than or equal to the 90th percentile for traffic proximity and is low income?,Traffic proximity and volume (percentile),traffic_prox,Greater than or equal to the 90th percentile for DOT transit barriers and is low income?,travel_barriers,Greater than or equal to the 90th percentile for housing burden and is low income?,Housing burden (percent) (percentile),housing_burden,"Greater than or equal to the 90th percentile for lead paint, the median house value is less than 90th percentile and is low income?",Percent pre-1960s housing (lead paint indicator) (percentile),Percent pre-1960s housing (lead paint indicator),Median value ($) of owner-occupied housing units (percentile),house_value,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent and is low income?,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent,Share of the tract's land area that is covered by impervious surface or cropland as a percent,pavement_cropland,Does the tract have at least 35 acres in it?,Tract experienced historic underinvestment and remains low income,Tract experienced historic underinvestment,no_plumbing,Share of homes with no kitchen or indoor plumbing (percent),Greater than or equal to the 90th percentile for proximity to hazardous waste facilities and is low income?,

In [11]:
# "'has_datacenter ~ ' + ' + '.join(predictors)"
#logistic regression model with fixed effects

#list of dummy dataframes
dummy_list = [linguistic_isolation_dummies, perc_poverty_dummies]

#making list of dummy columns that predict a data center ??
for new_dummy in dummy_list:
    #dummies that predict data center
    dummy_predictors = [i for i in new_dummy.columns if i != 'has_datacenter']

#combining regular predictor columns with dummy predictor columns
all_predictors = predictors + dummy_predictors
print(all_predictors)

#fixed effects model
my_fixed_effects_model = statsmodels.formula.api.logit('has_datacenter ~ ' + ' + '.join(all_predictors), 
    data=cejst, subset=None, drop_cols=None).fit(disp=False) #, *args, **kwargs

my_fixed_effects_model.summary()


['perc_white', 'perc_poverty', 'pm_air', 'perc_unemployment', 'hazard_waste_prox', 'life_expectancy', 'perc_hispanic', 'perc_elderly', 'perc_children', 'perc_disadvantaged', 'population', 'loss_rate', 'flood_risk', 'energy_burden', 'diesel_exposure', 'traffic_prox', 'travel_barriers', 'housing_burden', 'house_value', 'pavement_cropland', 'no_plumbing', 'superfund_prox', 'asthma', 'diabetes', 'heart_disease', 'household_income_compared', 'linguistic_isolation', 'no_hs_degree', 'perc_poverty_1', 'perc_poverty_2', 'perc_poverty_3', 'perc_poverty_4', 'perc_poverty_5', 'perc_poverty_7', 'perc_poverty_8', 'perc_poverty_9', 'perc_poverty_10', 'perc_poverty_11']


/Users/charlottemeyer/anaconda3/envs/.da_401/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:         has_datacenter   No. Observations:                59078
Model:                          Logit   Df Residuals:                    59039
Method:                           MLE   Df Model:                           38
Date:                Thu, 30 Apr 2026   Pseudo R-squ.:                 0.04832
Time:                        21:31:25   Log-Likelihood:                -1823.1
converged:                      False   LL-Null:                       -1915.7
Covariance Type:            nonrobust   LLR p-value:                 3.024e-21
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -1.7958      2.225     -0.807      0.420      -6.157       2.565
perc_poverty_1[T.True]        0.6221      1.023      0.608      0.543      -1.383       2.627
perc_poverty_2[T.True]        0.0299      0.859      0.035      0.972      -1.654       1.714
perc_poverty_3[T.True]        0.1623      0.672      0.242      0.809      -1.155       1.479
perc_poverty_4[T.True]        0.1244      0.487      0.255      0.799      -0.831       1.080
perc_poverty_5[T.True]       -0.5268      0.348     -1.512      0.131      -1.210       0.156
perc_poverty_7[T.True]       -0.0600      0.384     -0.156      0.876      -0.812       0.692
perc_poverty_8[T.True]       -0.0538      0.597     -0.090      0.928      -1.225       1.117
perc_poverty_9[T.True]        0.3292      0.873      0.377      0.706      -1.382       2.041
perc_poverty_10[T.True]      -5.2058     21.725     -0.240      0.811     -47.786      37.374
perc_poverty_11[T.True]     -13.8754   7145.720     -0.002      0.998    -1.4e+04     1.4e+04
perc_white                   -0.3747      0.504     -0.744      0.457      -1.362       0.612
perc_poverty                 -0.0398      2.489     -0.016      0.987      -4.918       4.838
pm_air                       -0.0395      0.040     -0.989      0.322      -0.118       0.039
perc_unemployment             0.0066      0.021      0.311      0.756      -0.035       0.048
hazard_waste_prox             0.0271      0.017      1.613      0.107      -0.006       0.060
life_expectancy               0.0059      0.022      0.268      0.789      -0.037       0.049
perc_hispanic                 0.1950      0.507      0.384      0.701      -0.800       1.190
perc_elderly                 -3.7759      1.747     -2.161      0.031      -7.200      -0.352
perc_children                -1.3376      1.779     -0.752      0.452      -4.824       2.149
perc_disadvantaged            0.0033      0.002      1.396      0.163      -0.001       0.008
population                -7.052e-05   2.99e-05     -2.361      0.018      -0.000    -1.2e-05
loss_rate                    34.5753     45.102      0.767      0.443     -53.822     122.973
flood_risk                   -0.0060      0.004     -1.360      0.174      -0.015       0.003
energy_burden                -0.4081      0.077     -5.322      0.000      -0.558      -0.258
diesel_exposure              -0.2765      0.358     -0.773      0.440      -0.978       0.425
traffic_prox               1.925e-05   3.32e-05      0.580      0.562   -4.58e-05    8.43e-05
travel_barriers              -0.0019      0.003     -0.717      0.473      -0.007       0.003
housing_burden               -0.0207      0.010     -2.147      0.032      -0.040      -0.002
house_value                -1.42e-06    4.2e-07     -3.384      0.001   -2.24e-06   -5.97e-07
pavement_cropland             0.0020      0.003      0.704      0.482      -0.004       0.008
no_plumbing                   0.0944      0.2

In [12]:
#list of columns that had low p-values in logit model
feature_columns_significant = ["perc_poverty",
                               "perc_hispanic",
                               "perc_elderly",
                               "population",
                               "loss_rate",
                               "travel_barriers",
                               "house_value",
                               "risk_facilities_prox",
                               "leaky_storage_tanks",
                               "asthma"]

new_significant = ["population", 
                   "energy_burden",
                   "risk_facilities_prox",
                   "linguistic_isolation"]
                   

In [14]:
#splittling into testing and training
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, test_size=0.25, random_state=26
#)

In [15]:
#still need this for the multicollinearity test
X = cejst[predictors]
y = cejst["has_datacenter"]

#dropping all rows with NaN in X columns
X = X.dropna()
#Keep y aligned with X
y = y.loc[X.index]  

# Verify they match
print(f"X shape: {X.shape}, y shape: {y.shape}")
#verify no na's
print(X.isnull().sum())

X shape: (59078, 28), y shape: (59078,)
perc_white                   0
perc_poverty                 0
pm_air                       0
perc_unemployment            0
hazard_waste_prox            0
life_expectancy              0
perc_hispanic                0
perc_elderly                 0
perc_children                0
perc_disadvantaged           0
population                   0
loss_rate                    0
flood_risk                   0
energy_burden                0
diesel_exposure              0
traffic_prox                 0
travel_barriers              0
housing_burden               0
house_value                  0
pavement_cropland            0
no_plumbing                  0
superfund_prox               0
asthma                       0
diabetes                     0
heart_disease                0
household_income_compared    0
linguistic_isolation         0
no_hs_degree                 0
dtype: int64


In [16]:
#code adapted from https://www.statology.org/how-to-test-for-multicollinearity-with-statsmodels/
#testing for multicollinearity

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Compute VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(X.shape[1])]

print(vif_data)

                      feature         VIF
0                  perc_white   28.306099
1                perc_poverty   36.799589
2                      pm_air   33.984501
3           perc_unemployment    4.468159
4           hazard_waste_prox    2.869737
5             life_expectancy  169.153638
6               perc_hispanic    5.166315
7                perc_elderly   20.266485
8               perc_children   15.103475
9          perc_disadvantaged    4.392691
10                 population    6.210092
11                  loss_rate    1.216665
12                 flood_risk    1.644375
13              energy_burden   10.629757
14            diesel_exposure    6.446378
15               traffic_prox    1.722776
16            travel_barriers    6.186916
17             housing_burden   18.805930
18                house_value    5.079448
19          pavement_cropland    7.096850
20                no_plumbing    4.452115
21             superfund_prox    1.318499
22                     asthma   11